In [1]:
# 📦 Step 0. Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 🔷 Step 1. Data Collection
df = pd.read_csv('ci_cd_logs.csv')
print("\n✅ Data Loaded Successfully")
print(df.head())
print(df.info())
print(df['status'].value_counts())



✅ Data Loaded Successfully
                  timestamp pipeline_id stage_name           job_name  \
0  2024-03-02 01:05:07+0000  pipe-txnem      Build  deploy_to_staging   
1  2024-07-22 19:55:41+0000  pipe-hjahz      Build     run_unit_tests   
2  2024-03-01 23:03:43+0000  pipe-vcsbx   Analysis      deploy_to_dev   
3  2024-06-02 12:21:00+0000  pipe-pnvzk       Test      deploy_to_dev   
4  2024-04-17 07:59:29+0000  pipe-mwkkl       Test     build_and_test   

  task_name   status                                       message  \
0   analyze  success                  Task completed successfully.   
1    deploy  skipped  Task was skipped due to pipeline conditions.   
2    deploy  success                  Task completed successfully.   
3      test  skipped  Task was skipped due to pipeline conditions.   
4      test   failed                        Task execution failed.   

                                  commit_id      branch      user environment  
0  f831dbe56dcbceadccc1447923b1d

In [2]:
df = pd.read_csv('ci_cd_logs.csv')

print(df.head())
print(df.info())
print(df['status'].value_counts())  # Assuming 'status' is the target column with Pass/Fail


                  timestamp pipeline_id stage_name           job_name  \
0  2024-03-02 01:05:07+0000  pipe-txnem      Build  deploy_to_staging   
1  2024-07-22 19:55:41+0000  pipe-hjahz      Build     run_unit_tests   
2  2024-03-01 23:03:43+0000  pipe-vcsbx   Analysis      deploy_to_dev   
3  2024-06-02 12:21:00+0000  pipe-pnvzk       Test      deploy_to_dev   
4  2024-04-17 07:59:29+0000  pipe-mwkkl       Test     build_and_test   

  task_name   status                                       message  \
0   analyze  success                  Task completed successfully.   
1    deploy  skipped  Task was skipped due to pipeline conditions.   
2    deploy  success                  Task completed successfully.   
3      test  skipped  Task was skipped due to pipeline conditions.   
4      test   failed                        Task execution failed.   

                                  commit_id      branch      user environment  
0  f831dbe56dcbceadccc1447923b1d9659becadbe  branch_ajn     

In [3]:
# Fill missing numeric values with median
df.fillna(df.median(numeric_only=True), inplace=True)

# Fill remaining missing (categorical) with mode
df.fillna(df.mode().iloc[0], inplace=True)

# Encode categorical variables
label_encoders = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("\n✅ Data Preprocessing Completed. No missing values remain.")



✅ Data Preprocessing Completed. No missing values remain.


In [4]:
# Build Duration Feature
if 'start_time' in df.columns and 'end_time' in df.columns:
    df['build_duration'] = pd.to_datetime(df['end_time']) - pd.to_datetime(df['start_time'])
    df['build_duration'] = df['build_duration'].dt.total_seconds()
else:
    # Replace dummy with parsed duration logic in production
    df['build_duration'] = np.random.randint(100, 1000, size=len(df))

# Number of Files Changed
if 'num_files_changed' in df.columns:
    df['num_files_changed'] = df['num_files_changed']
else:
    # Replace dummy with git diff parsing in production
    df['num_files_changed'] = np.random.randint(1, 20, size=len(df))

# Test Pass Rate
if 'tests_passed' in df.columns and 'tests_run' in df.columns:
    df['test_pass_rate'] = df['tests_passed'] / df['tests_run']
else:
    # Replace dummy with parsed test suite results in production
    df['test_pass_rate'] = np.random.uniform(0.7, 1.0, size=len(df))

# Previous Build Status (lag feature)
df['prev_status'] = df['status'].shift(1).fillna(df['status'].mode()[0])
le_prev = LabelEncoder()
df['prev_status_encoded'] = le_prev.fit_transform(df['prev_status'])

print("\n✅ Feature Engineering Completed.")
print(df[['build_duration', 'num_files_changed', 'test_pass_rate', 'prev_status_encoded']].head())



✅ Feature Engineering Completed.
   build_duration  num_files_changed  test_pass_rate  prev_status_encoded
0             446                  1        0.977464                    2
1             270                 11        0.780947                    3
2             612                 13        0.738410                    2
3             983                  2        0.840358                    3
4             192                 12        0.875910                    2


In [5]:
# Encode target
le_target = LabelEncoder()
df['status_encoded'] = le_target.fit_transform(df['status'])

# Define X and y
X = df.drop(['status', 'status_encoded', 'prev_status'], axis=1)
y = df['status_encoded']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("\n✅ Train-Test Split Completed")
print("Training samples:", X_train.shape[0], "Testing samples:", X_test.shape[0])

# Logistic Regression
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("\n🔷 Logistic Regression Performance")
print(classification_report(y_test, lr_preds, target_names=[str(c) for c in le_target.classes_]))
print("Accuracy:", accuracy_score(y_test, lr_preds))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("\n🔷 Random Forest Performance")
print(classification_report(y_test, rf_preds, target_names=[str(c) for c in le_target.classes_]))
print("Accuracy:", accuracy_score(y_test, rf_preds))



✅ Train-Test Split Completed
Training samples: 640 Testing samples: 160

🔷 Logistic Regression Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        39
           1       1.00      1.00      1.00        41
           2       1.00      1.00      1.00        44
           3       1.00      1.00      1.00        36

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160

Accuracy: 1.0

🔷 Random Forest Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        39
           1       1.00      1.00      1.00        41
           2       1.00      1.00      1.00        44
           3       1.00      1.00      1.00        36

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00 

In [6]:
# Decode predictions and actuals
y_test_decoded = le_target.inverse_transform(y_test)
rf_preds_decoded = le_target.inverse_transform(rf_preds)

# Create comparison DataFrame
results_df = pd.DataFrame({
    'Actual': y_test_decoded,
    'Predicted': rf_preds_decoded
})
print("\n🔷 Predicted vs Actual Comparison")
print(results_df.head(20))



🔷 Predicted vs Actual Comparison
    Actual  Predicted
0        2          2
1        1          1
2        1          1
3        3          3
4        2          2
5        1          1
6        2          2
7        0          0
8        0          0
9        3          3
10       0          0
11       1          1
12       2          2
13       0          0
14       0          0
15       1          1
16       0          0
17       2          2
18       0          0
19       3          3


In [7]:
# Feature Importances
importances = pd.Series(rf.feature_importances_, index=X.columns)
important_features = importances.sort_values(ascending=False)

print("\n🔷 Top Features and Recommendations")
for feature, importance in important_features.items():
    if importance > 0.05:
        if 'file_change' in feature:
            print(f"- High {feature}: Review large code changes to reduce build failures.")
        elif 'developer' in feature:
            print(f"- Developer pattern: Frequent failures by certain developers. Encourage code reviews or pair programming.")
        elif 'build_duration' in feature:
            print(f"- Long build durations increase failure risk. Optimize build steps or caching strategies.")
        else:
            print(f"- {feature}: Important factor with impact score {importance:.2f}")



🔷 Top Features and Recommendations
- message: Important factor with impact score 0.63


In [8]:
# 📦 🔷 Step 8: Failure Prediction Alert System

# View expected feature order
print("\n🔎 Required feature columns (order-sensitive):", list(X.columns))

# 🔹 Create new_build with dummy values for all features
# Replace these with real upcoming build data as needed

new_build_dict = {}
for col in X.columns:
    if col in ['build_duration']:
        new_build_dict[col] = [500]
    elif col in ['num_files_changed']:
        new_build_dict[col] = [10]
    elif col in ['test_pass_rate']:
        new_build_dict[col] = [0.85]
    elif col in ['prev_status_encoded']:
        new_build_dict[col] = [1]
    else:
        new_build_dict[col] = [0]  # default dummy for other categorical/encoded features

# Convert to DataFrame with same column order
new_build = pd.DataFrame(new_build_dict)

# ✅ Confirm matching columns and order
assert list(new_build.columns) == list(X.columns), "Column order mismatch."

# Scale new build data
new_build_scaled = scaler.transform(new_build)

# Predict failure probability
failure_prob = rf.predict_proba(new_build_scaled)[0][1]

print("\n🔷 🔔 Build Failure Prediction Alert System")
print(f"Predicted Failure Probability: {failure_prob:.2f}")

threshold = 0.5

if failure_prob >= threshold:
    print("\n🚨 ALERT: Build likely to FAIL. Suggested Preventive Actions:")
    
    importances = pd.Series(rf.feature_importances_, index=X.columns)
    important_features = importances.sort_values(ascending=False)
    
    for feature, importance in important_features.items():
        if importance > 0.05:
            if 'file_change' in feature:
                print(f"- High {feature}: Review large code changes to reduce build failures.")
            elif 'developer' in feature:
                print(f"- Developer pattern: Frequent failures by this developer. Encourage code reviews or pair programming.")
            elif 'build_duration' in feature:
                print(f"- Long build durations increase failure risk. Optimize build steps or caching strategies.")
            elif 'test_pass_rate' in feature:
                print(f"- Low test pass rates. Improve unit and integration test coverage before merging.")
            else:
                print(f"- {feature}: High impact factor (score {importance:.2f}).")
else:
    print("\n✅ Build predicted to PASS. Proceed with pipeline execution.")



🔎 Required feature columns (order-sensitive): ['timestamp', 'pipeline_id', 'stage_name', 'job_name', 'task_name', 'message', 'commit_id', 'branch', 'user', 'environment', 'build_duration', 'num_files_changed', 'test_pass_rate', 'prev_status_encoded']

🔷 🔔 Build Failure Prediction Alert System
Predicted Failure Probability: 0.23

✅ Build predicted to PASS. Proceed with pipeline execution.
